# Screened-data unlearning experiment
Run setup and saved analysis to inspect the submitted results without a GPU.
For fresh inference, select a GPU runtime and enable the Colab `HF_TOKEN` secret.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/content/unlearning_task')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LinaGolan/unlearning_task.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

def run(*args):
    subprocess.run([sys.executable, '-m', *map(str, args)], check=True)


## Saved analysis — no inference

In [ ]:
run('unlearning', 'report')
for k in (2, 4):
    out = Path(f'results_fullscreen_experiment/k{k}')
    run('unlearning', 'report', '--methods', 'gate', '--out', out, '--figures', out / 'figures')
from IPython.display import HTML, display
display(HTML(Path('Research_report_fullscreen.html').read_text()))


## Fresh inference — optional
The saved 512 WMDP, 512 retain and 64 biology questions are the default.
Results go to Drive; rerun these cells to resume completed conditions after a disconnect.
Change `work` for a new independent run. No model weights are trained.


In [ ]:
from google.colab import drive, userdata
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime first.'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
work = Path('/content/drive/MyDrive/unlearning_screened_run')
work.mkdir(parents=True, exist_ok=True)
data = repo / 'results_fullscreen/selected_data'
config = repo / 'results_fullscreen/selected_config.json'


### Optional: repeat screening from scratch
Leave `rescreen = False` to use the submitted subset. Fresh screening checks original
answers first, then all remaining orderings only for correct questions, and keeps
those correct in at least 18/24. It may produce a different subset.


In [ ]:
rescreen = False
if rescreen:
    source, screening = work / 'source', work / 'screening'
    if not (source / 'manifest.json').exists():
        run('unlearning.full_screening', 'prepare', '--source', source)
    run('unlearning.full_screening', 'run', '--source', source, '--out', screening, '--device', 'cuda')
    data, config = screening / 'selected_data', screening / 'selected_config.json'


### Main experiment — gate and gate margin

In [ ]:
out = work / 'main'
run('unlearning', 'run', '--config', config, '--data', data, '--out', out, '--device', 'cuda')
run('unlearning', 'report', '--out', out, '--figures', out / 'figures')


### Layer-budget controls — gate

In [ ]:
for k in (2, 4):
    out = work / f'k{k}'
    run('unlearning', 'run', '--config', config, '--data', data, '--out', out,
        '--device', 'cuda', '--methods', 'gate', '--k', k, '--strengths', '0,0.5,1.0')
    run('unlearning', 'report', '--methods', 'gate', '--out', out, '--figures', out / 'figures')
print('Results saved in', work)
